# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chapcoda/flyrank-ML-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Random Forest**

I'm using a Random Forest classifier, scored the same way as the baseline (rank by predicted probability, take the top 50, measure Precision@50 against `is_declining_label`).

Two things from my own findings point here specifically, not just "trees are usually fine":

1. **Signal Check 1 (Week 4) found a nonmonotonic, bucketed relationship** — visibility peaks hard in the 31–90 day staleness window, then collapses, rather than declining smoothly with age. A linear model like logistic regression can't represent a "peak in the middle" relationship without manual feature engineering (binning, interaction terms); a tree-based model finds that threshold structure on its own by splitting on `days_since_last_update` at whatever points the data actually supports.
2. **The label is close to balanced (50.2% declining)**, so there's no need for class-imbalance handling that would complicate the method choice — a plain Random Forest is appropriate without resampling or weighting.
Random Forest specifically (over Gradient Boosting) because: it's more robust to the smaller feature set here (CTR-vs-position was dropped entirely in Week 4 after being ruled FALSE, so there are fewer engineered signals to lean on), it's less prone to overfitting on a dataset this size split across only 29 clients, and it gives permutation importance for free, which Section 4 (error analysis) needs anyway.

In [1]:
# ============ SETUP (self-contained — safe to re-run any time) ============
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

DATASET = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{DATASET}/dim_content.parquet"

# Feature window = Feb 1 - Apr 30, 2026 (three month partitions, unioned)
FEAT_MONTHS = ["2026-02", "2026-03", "2026-04"]
FEAT_PATHS = [f"{DATASET}/fact_content_daily_performance/month={m}/data_0.parquet" for m in FEAT_MONTHS]
FEAT_UNION_SQL = " UNION ALL ".join([f"SELECT * FROM read_parquet('{p}')" for p in FEAT_PATHS])

print("Setup complete. Working dir:", os.getcwd())

# ============ FEATURE TABLE — Feb 1 - Apr 30 2026, per Week 3 data contract ============
q_features = f"""
WITH feature_window AS (
    {FEAT_UNION_SQL}
),
perf AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS impressions_90d,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_clicks ELSE 0 END) AS clicks_90d,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_avg_position * gsc_impressions ELSE 0 END)
            / NULLIF(SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END), 0)
            AS avg_position_weighted,
        SUM(CASE WHEN ga4_data_available = TRUE THEN ga4_sessions ELSE 0 END) AS sessions_90d,
        SUM(CASE WHEN ga4_data_available = TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_90d,
        SUM(CASE WHEN ga4_data_available = TRUE THEN ga4_total_engagement_sec ELSE 0 END) AS engagement_sec_90d,
        SUM(CASE WHEN ga4_data_available = TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d,
        BOOL_OR(gsc_data_available) AS any_gsc_available,
        BOOL_OR(ga4_data_available) AS any_ga4_available
    FROM feature_window
    GROUP BY content_hash_id, client_hash_id
),
dims AS (
    SELECT
        content_hash_id,
        client_hash_id,
        content_type,
        main_intent,
        backlinks,
        char_count,
        word_count,
        search_volume,
        competition,
        competition_level,
        cpc,
        DATE_DIFF('day', content_created_date, DATE '2026-04-30') AS content_age_days,
        DATE_DIFF('day', content_updated_date, DATE '2026-04-30') AS days_since_last_update,
        DATE_DIFF('day', last_optimized_date, DATE '2026-04-30') AS days_since_last_optimized
    FROM read_parquet('{DIM_CONTENT}')
    WHERE is_published = TRUE
      AND is_deleted = FALSE
)
SELECT
    d.content_hash_id,
    d.client_hash_id,
    d.content_type,
    d.main_intent,
    d.backlinks,
    d.char_count,
    d.word_count,
    d.search_volume,
    d.competition,
    d.competition_level,
    d.cpc,
    d.content_age_days,
    d.days_since_last_update,
    d.days_since_last_optimized,
    COALESCE(p.impressions_90d, 0) AS impressions_90d,
    COALESCE(p.clicks_90d, 0) AS clicks_90d,
    p.avg_position_weighted,
    COALESCE(p.sessions_90d, 0) AS sessions_90d,
    COALESCE(p.engaged_sessions_90d, 0) AS engaged_sessions_90d,
    COALESCE(p.engagement_sec_90d, 0) AS engagement_sec_90d,
    COALESCE(p.sessions_organic_90d, 0) AS sessions_organic_90d,
    COALESCE(p.any_gsc_available, FALSE) AS any_gsc_available,
    COALESCE(p.any_ga4_available, FALSE) AS any_ga4_available
FROM dims d
LEFT JOIN perf p ON d.content_hash_id = p.content_hash_id
WHERE d.days_since_last_update >= 0
"""

features = con.execute(q_features).fetchdf()

# Derived ratios (guard divide-by-zero)
features["ctr"] = np.where(
    features["impressions_90d"] > 0,
    features["clicks_90d"] / features["impressions_90d"],
    0.0,
)
features["engagement_rate"] = np.where(
    features["sessions_90d"] > 0,
    features["engaged_sessions_90d"] / features["sessions_90d"],
    0.0,
)

print(f"Feature table: {len(features):,} rows, {features['client_hash_id'].nunique()} clients")
print(f"GSC-honest coverage: {features['any_gsc_available'].mean():.1%}")
print(f"GA4-honest coverage: {features['any_ga4_available'].mean():.1%}")
features.head(10)

Setup complete. Working dir: /content/flyrank-ml-internship-starter


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature table: 50,840 rows, 37 clients
GSC-honest coverage: 88.3%
GA4-honest coverage: 43.9%


,content_hash_id,client_hash_id,content_type,main_intent,backlinks,char_count,word_count,search_volume,competition,competition_level,...,clicks_90d,avg_position_weighted,sessions_90d,engaged_sessions_90d,engagement_sec_90d,sessions_organic_90d,any_gsc_available,any_ga4_available,ctr,engagement_rate
0,content_99b5d627ad600bc0,client_e547b89c05043229,keyword article,transactional,<NA>,11663,1739,10,0.80,HIGH,...,1.0,40.270531,0.0,0.0,0.0,0.0,True,False,0.004831,0.000000
1,content_f13729777cc7cd45,client_e547b89c05043229,keyword article,transactional,<NA>,9023,1379,10,0.00,LOW,...,1.0,16.667297,0.0,0.0,0.0,0.0,True,False,0.001890,0.000000
2,content_5bcf50ce63a63eeb,client_e547b89c05043229,keyword article,commercial,<NA>,10758,1639,10,0.29,LOW,...,3.0,9.107463,2.0,0.0,0.0,1.0,True,True,0.004478,0.000000
3,content_c29f69d03f3c6c4f,client_e547b89c05043229,keyword article,informational,<NA>,8794,1359,30,0.00,LOW,...,0.0,7.165049,2.0,0.0,0.0,0.0,True,True,0.000000,0.000000
4,content_4dfb073b6dac6f8e,client_e547b89c05043229,keyword article,transactional,<NA>,11768,1828,10,0.00,LOW,...,2.0,24.702006,6.0,1.0,298.0,2.0,True,True,0.005731,0.166667
5,content_a7d8566bc307fd91,client_e547b89c05043229,keyword article,informational,<NA>,10145,1527,10,0.00,LOW,...,0.0,29.206897,0.0,0.0,0.0,0.0,True,False,0.000000,0.000000
6,content_27f50fc7c8b4b12b,client_e547b89c05043229,keyword article,informational,<NA>,9608,1425,20,0.06,LOW,...,3.0,6.126400,8.0,0.0,0.0,3.0,True,True,0.004800,0.000000
7,content_39b8a061f7ed098d,client_e547b89c05043229,keyword article,informational,<NA>,11802,1781,10,0.35,MEDIUM,...,0.0,42.254545,0.0,0.0,0.0,0.0,True,False,0.000000,0.000000
8,content_2006a2b84229122b,client_e547b89c05043229,keyword article,informational,<NA>,11115,1658,10,0.00,LOW,...,0.0,6.551402,0.0,0.0,0.0,0.0,True,False,0.000000,0.000000
9,content_10f6a9ecff5f1c42,client_e547b89c05043229,keyword article,informational,<NA>,11468,1711,10,0.49,MEDIUM,...,1.0,28.076775,2.0,0.0,0.0,1.0,True,True,0.000960,0.000000


## 2. Split design

**Split type: client-holdout (grouped by `client_hash_id`), plus a time-aware label**

The split has two layers, and both matter for different reasons.

**1. Grouped by client, not by row.** I used `GroupShuffleSplit` on `client_hash_id` (80/20, `random_state=42`) so that every page belonging to a given client lands entirely in train or entirely in test — never split across both. Result: 30,819 rows across 23 clients in train, 2,477 rows across 6 clients in test, with zero client overlap (confirmed by set intersection, not assumed).

This is the honest choice for this question because pages from the same client aren't independent of each other — they share the same CMS, editorial cadence, backlink profile, content templates, and general traffic level. A plain random row-level split would let pages from the same client appear in both train and test, and the model could partly "solve" the task by memorizing client-level patterns (e.g. "pages from client X tend to decline") rather than learning genuine page-level decline signals. That would produce an inflated, dishonest test score — good performance on clients the model has effectively already seen, not evidence it generalizes to a new client's content. Since the real use case is scoring pages for content teams across many different clients, including ones not in the training data, holding out entire clients is the split that actually tests what matters.

**2. Time-aware by construction, not by explicit train/test date cutoff.** The feature window (Feb 1 – Apr 30, 2026) and the label's target window (May 1–31, 2026) never overlap — this is enforced upstream in how `model_df` was built, not something the split itself needs to re-enforce. Every row, in both train and test, uses only pre-May data as features and only May data for the label. So there's no separate "time split" layered on top of the client split; leakage-safety across time is guaranteed by the feature/label construction itself, and the client split only needs to guard against client-identity leakage, which is the risk a time-based split alone wouldn't catch.

**Checked, not assumed:** train declining rate 50.2%, test declining rate 50.6% — close enough that the split didn't accidentally skew the label balance between train and test, which would have made the comparison harder to interpret.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# ============ LABEL — is_declining_label, per Week 3 data contract ============
# Definition: 1 if a page's May 2026 gsc_impressions are at least 25% lower than its
# trailing 30-day impressions from the feature window (April 2026); else 0.
# Trailing 30d = the April partition (already a full month, so it doubles as "last 30 days
# of the feature window" cleanly). Target = May partition. No overlap with Feb-Apr features.

TRAILING_PATH = f"{DATASET}/fact_content_daily_performance/month=2026-04/data_0.parquet"
TARGET_PATH = f"{DATASET}/fact_content_daily_performance/month=2026-05/data_0.parquet"

q_label = f"""
WITH trailing_30d AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS trailing_impressions
    FROM read_parquet('{TRAILING_PATH}')
    GROUP BY content_hash_id
),
target_window AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS target_impressions
    FROM read_parquet('{TARGET_PATH}')
    GROUP BY content_hash_id
)
SELECT
    t.content_hash_id,
    t.trailing_impressions,
    COALESCE(g.target_impressions, 0) AS target_impressions,
    CASE
        WHEN t.trailing_impressions > 0
            THEN CASE WHEN COALESCE(g.target_impressions, 0) <= t.trailing_impressions * 0.75
                      THEN 1 ELSE 0 END
        ELSE NULL  -- page had zero trailing impressions: "decline" is undefined, not "no decline"
    END AS is_declining_label
FROM trailing_30d t
LEFT JOIN target_window g ON t.content_hash_id = g.content_hash_id
"""

labels = con.execute(q_label).fetchdf()

n_total = len(labels)
n_excluded = labels["is_declining_label"].isna().sum()
n_labeled = n_total - n_excluded

print(f"Pages with a trailing-30d impression count: {n_total:,}")
print(f"Excluded (zero trailing impressions, label undefined): {n_excluded:,}")
print(f"Labeled pages: {n_labeled:,}")
print(f"Declining rate among labeled pages: {labels['is_declining_label'].mean(skipna=True):.1%}")

# ============ JOIN LABEL ONTO FEATURE TABLE ============
model_df = features.merge(labels[["content_hash_id", "is_declining_label"]], on="content_hash_id", how="inner")
model_df = model_df.dropna(subset=["is_declining_label"]).copy()
model_df["is_declining_label"] = model_df["is_declining_label"].astype(int)

print(f"\nFinal modeling table: {len(model_df):,} rows, {model_df['client_hash_id'].nunique()} clients")
print(f"Declining rate: {model_df['is_declining_label'].mean():.1%}")
model_df["is_declining_label"].value_counts()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with a trailing-30d impression count: 362,172
Excluded (zero trailing impressions, label undefined): 167,412
Labeled pages: 194,760
Declining rate among labeled pages: 47.6%

Final modeling table: 33,296 rows, 29 clients
Declining rate: 50.2%


,count
is_declining_label,
1,16711
0,16585


In [3]:
# ============ SECTION 2 — SPLIT DESIGN (client-holdout) ============
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(model_df, groups=model_df["client_hash_id"]))

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])
overlap = train_clients & test_clients

print(f"Train: {len(train_df):,} rows, {len(train_clients)} clients")
print(f"Test:  {len(test_df):,} rows, {len(test_clients)} clients")
print(f"Client overlap between train/test: {len(overlap)}  (must be 0)")
print(f"Train declining rate: {train_df['is_declining_label'].mean():.1%}")
print(f"Test declining rate:  {test_df['is_declining_label'].mean():.1%}")


Train: 30,819 rows, 23 clients
Test:  2,477 rows, 6 clients
Client overlap between train/test: 0  (must be 0)
Train declining rate: 50.2%
Test declining rate:  50.6%


## 3. Train + compare vs my baseline

**Same data, same split, same metric.** The baseline rule (staleness-weight × log-impressions, identical logic to `w04_baseline_score.ipynb`) was recomputed directly on the held-out test set from Section 2 — not pulled from the old CSV — so both methods are scored on the exact same 2,477 rows, the same client-holdout test split, and the same metric (Precision@50 against `is_declining_label`).

| method | precision_at_50 | roc_auc | avg_precision |
|---|---|---|---|
| baseline_rule (w04) | 0.260 | 0.298 | 0.388 |
| random_forest | 0.520 | 0.634 | 0.605 |

Test set: 2,477 rows, 50.6% declining.

**Reading the table honestly:** the baseline's ROC AUC (0.298) is *below* 0.5 — worse than random. With the test set at 50.6% declining, a random ranking would expect ~50% Precision@50 by chance; the baseline scored 26%, roughly half of that. The staleness×visibility rule isn't just weak here, it's pointing in the wrong direction relative to `is_declining_label` — it tends to flag pages that are currently visible and moderately stale, which describes pages doing fine right now more than pages about to lose visibility in May.

The Random Forest clearly beats the baseline (0.660 vs. 0.260 — 2.5x), and its AUC (0.634) shows real signal above chance. Against the test set's 50.6% base rate, 0.660 means 33 of the top 50 were genuinely declining versus the ~25 random picking would give — a real margin at the top of the ranking, which is the part that matters for a reviewer working a queue downward. Worth noting the honest limit: an AUC of 0.634 is only moderate, so the model discriminates well among the highest-scoring pages but much less sharply across the full distribution. It's a decisive improvement over a broken baseline and useful at the top of the queue, not a strong classifier everywhere. Full breakdown of why in Section 4.

In [4]:
# ============ SECTION 3 — TRAIN + COMPARE VS BASELINE ============
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_features = [
    "char_count", "word_count", "search_volume", "competition", "cpc",
    "content_age_days", "days_since_last_update", "days_since_last_optimized",
    "impressions_90d", "clicks_90d", "avg_position_weighted",
    "sessions_90d", "engaged_sessions_90d", "engagement_sec_90d",
    "sessions_organic_90d", "ctr", "engagement_rate",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )),
])

X_train, y_train = train_df[numeric_features + categorical_features], train_df["is_declining_label"]
X_test, y_test = test_df[numeric_features + categorical_features], test_df["is_declining_label"]

model.fit(X_train, y_train)
test_df = test_df.copy()
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

# --- Baseline rule, recomputed on this SAME test set (same logic as w04) ---
bucket_weights = {
    "1) 0-30d":    9.8 / 51.6,
    "2) 31-90d":   51.6 / 51.6,
    "3) 91-180d":  3.4 / 51.6,
    "4) 181-365d": 0.4 / 51.6,
    "5) 365+d":    0.1 / 51.6,
}
def age_bucket(days):
    if days <= 30: return "1) 0-30d"
    if days <= 90: return "2) 31-90d"
    if days <= 180: return "3) 91-180d"
    if days <= 365: return "4) 181-365d"
    return "5) 365+d"

test_df["age_bucket"] = test_df["days_since_last_update"].apply(age_bucket)
test_df["staleness_weight"] = test_df["age_bucket"].map(bucket_weights)
test_df["baseline_score"] = test_df["staleness_weight"] * np.log1p(test_df["impressions_90d"])

# --- Precision@50, both scored on the identical test set ---
def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k["is_declining_label"].mean(), top_k

baseline_p50, baseline_top50 = precision_at_k(test_df, "baseline_score", 50)
model_p50, model_top50 = precision_at_k(test_df, "model_score", 50)

baseline_auc = roc_auc_score(test_df["is_declining_label"], test_df["baseline_score"])
model_auc = roc_auc_score(test_df["is_declining_label"], test_df["model_score"])
baseline_ap = average_precision_score(test_df["is_declining_label"], test_df["baseline_score"])
model_ap = average_precision_score(test_df["is_declining_label"], test_df["model_score"])

comparison = pd.DataFrame({
    "method": ["baseline_rule (w04)", "random_forest"],
    "precision_at_50": [round(baseline_p50, 3), round(model_p50, 3)],
    "roc_auc": [round(baseline_auc, 3), round(model_auc, 3)],
    "avg_precision": [round(baseline_ap, 3), round(model_ap, 3)],
})

print(f"\nTest set: {len(test_df):,} rows, {test_df['is_declining_label'].mean():.1%} declining")
print(f"\nBaseline rule  Precision@50: {baseline_p50:.3f}  (~{round(baseline_p50*50)} of top 50 right)")
print(f"Random forest  Precision@50: {model_p50:.3f}  (~{round(model_p50*50)} of top 50 right)")
comparison

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['days_since_last_optimized']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['days_since_last_optimized']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



Test set: 2,477 rows, 50.6% declining

Baseline rule  Precision@50: 0.260  (~13 of top 50 right)
Random forest  Precision@50: 0.660  (~33 of top 50 right)


,method,precision_at_50,roc_auc,avg_precision
0,baseline_rule (w04),0.26,0.298,0.388
1,random_forest,0.66,0.630,0.606


## 4. Errors and interpretation

**What the model leans on**

Permutation importance is dominated by two features: `avg_position_weighted` (0.043) and `ctr` (0.031), with `clicks_90d` a distant third (0.015). Everything else — including `days_since_last_update`, the feature the entire Week 4 baseline rule was built around — contributes nothing measurable (-0.0004; shuffling it if anything nudged AUC up slightly, which is what a feature carrying no signal looks like).
That's worth reconciling honestly against Week 4's own Signal Check 2, which ruled CTR-vs-position `FALSE` and dropped it — but that verdict was based on a narrow diagnostic (the `page_1` tier specifically, using March-only daily `avg_position` values, some of which came back nonsensically below 1). Here, `avg_position_weighted` is computed as an impressions-weighted average across three months and lands in a plausible 7–51 range with no fractional-below-1 artifacts. The two findings aren't necessarily contradictory — the earlier anomaly may have been isolated to a specific position tier or a daily-granularity quirk — but it means the CTR/position signal deserves a second look rather than staying permanently discarded. I'm flagging the tension rather than resolving it here.

**Where the model is wrong: false positives (top 50)**

Pages the model flagged as high-risk but that were actually stable had roughly half the impressions of true positives (361 vs. 727) and far fewer clicks (0.06 vs. 0.21).

**Where the model is wrong: false negatives (bottom 50)**

More concerning is the blind spot: pages the model scored as lowest-risk but that were actually declining had a much worse average position (54.5 vs. 30.4 for true negatives), a quarter of the impressions (417 vs. 1,769), and no clicks at all (0.0 vs. 5.1).

**Baseline vs. model overlap**

Only 1 of the baseline's top 50 and the model's top 50 pages is the same.

**Bottom line:** the Random Forest clearly outperforms the baseline rule on Precision@50 (0.660 vs. 0.260) and does so by finding a different, apparently more relevant signal than the one the hand-written rule relied on. The gain is real where it counts — 33 of the top 50 correct against a 50.6% base rate — but the moderate AUC (0.634) says the model separates well at the top of the ranking and much less sharply across the rest of the distribution. And its specific failure mode (missing already-low-visibility pages that keep declining) is exactly the blind spot a content team would care most about avoiding.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# ============ SECTION 4 — ERRORS AND INTERPRETATION ============
from sklearn.inspection import permutation_importance

# --- What does the model actually lean on? ---
perm = permutation_importance(
    model, X_test, y_test, scoring="roc_auc",
    n_repeats=10, random_state=42, n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

print("Permutation importance (drop in ROC AUC when feature is shuffled — higher = more relied on):")
display(importance_df.head(10))
# Note: days_since_last_optimized should show ~0 importance — it's entirely empty in this
# warehouse slice (flagged by the imputer warning in Section 3), so the model can't be
# leaning on it no matter what this number says.

# --- Where does the model's top-50 go wrong? ---
model_hits = int(model_top50["is_declining_label"].sum())
model_misses = 50 - model_hits
print(f"\nModel's top 50: {model_hits} correctly declining, {model_misses} false positives")

compare_cols = ["impressions_90d", "clicks_90d", "avg_position_weighted",
                "sessions_90d", "days_since_last_update", "ctr"]

false_positives = model_top50[model_top50["is_declining_label"] == 0]
true_positives = model_top50[model_top50["is_declining_label"] == 1]

print("\nFalse positives vs true positives — mean feature values within the model's top 50:")
display(pd.DataFrame({
    "false_positive_mean": false_positives[compare_cols].mean(),
    "true_positive_mean": true_positives[compare_cols].mean(),
}))

# --- What is the model missing? (declining pages it ranked as LOW risk) ---
bottom50 = test_df.sort_values("model_score", ascending=True).head(50)
false_negatives = bottom50[bottom50["is_declining_label"] == 1]
true_negatives = bottom50[bottom50["is_declining_label"] == 0]

print(f"\nModel's bottom 50 (lowest predicted risk): {len(false_negatives)} were actually declining (missed)")
print("\nFalse negatives vs true negatives — mean feature values within the model's bottom 50:")
display(pd.DataFrame({
    "false_negative_mean": false_negatives[compare_cols].mean(),
    "true_negative_mean": true_negatives[compare_cols].mean(),
}))

# --- How much does the model's top 50 even resemble the baseline's? ---
overlap_ids = set(baseline_top50["content_hash_id"]) & set(model_top50["content_hash_id"])
print(f"\nPages shared between baseline's top 50 and model's top 50: {len(overlap_ids)} / 50")

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['days_since_last_optimized']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Permutation importance (drop in ROC AUC when feature is shuffled — higher = more relied on):


,feature,importance_mean,importance_std
10,avg_position_weighted,0.042769,0.002921
15,ctr,0.030524,0.002317
9,clicks_90d,0.015182,0.001570
14,sessions_organic_90d,0.004282,0.000853
11,sessions_90d,0.002197,0.001439
13,engagement_sec_90d,0.000271,0.001034
7,days_since_last_optimized,0.000000,0.000000
12,engaged_sessions_90d,-0.000254,0.000353
6,days_since_last_update,-0.000400,0.000482
2,search_volume,-0.000421,0.002047



Model's top 50: 33 correctly declining, 17 false positives

False positives vs true positives — mean feature values within the model's top 50:


,false_positive_mean,true_positive_mean
impressions_90d,361.294118,726.606061
clicks_90d,0.058824,0.212121
avg_position_weighted,10.230751,7.462563
sessions_90d,1.764706,2.393939
days_since_last_update,62.941176,61.393939
ctr,0.000865,0.000035



Model's bottom 50 (lowest predicted risk): 8 were actually declining (missed)

False negatives vs true negatives — mean feature values within the model's bottom 50:


,false_negative_mean,true_negative_mean
impressions_90d,417.00000,1768.857143
clicks_90d,0.00000,5.119048
avg_position_weighted,54.52003,30.447505
sessions_90d,3.75000,11.428571
days_since_last_update,64.00000,64.000000
ctr,0.00000,0.002191



Pages shared between baseline's top 50 and model's top 50: 1 / 50


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.